# 02. Huấn luyện & Ablation Study cho BiLSTM-CRF (Kaggle / Colab)

Notebook của **Tín**: thiết lập môi trường trên Kaggle/Colab, tải dữ liệu `PhoNER_COVID19`, chạy mô hình `BiLSTM-CRF` đầy đủ (`CharCNN + BiLSTM + LinearChainCRF`) và các nhánh đối chứng (Ablation Study).

In [1]:
import os
from pathlib import Path

REPO_URL = "https://github.com/TTins1702/NER-PhoNER_COVID19-UET.git"
BRANCH = "main"
REPO_DIR = "NER-PhoNER_COVID19-UET"

def _is_repo_root(p: Path) -> bool:
    return (p / "scripts" / "train.py").exists() and (p / "src" / "data" / "labels.py").exists()

if Path.cwd() == Path("/kaggle") and Path("/kaggle/working").exists():
    %cd /kaggle/working
if _is_repo_root(Path("..")):
    %cd ..
if not _is_repo_root(Path(".")):
    if not Path(REPO_DIR).exists():
        !git clone -b {BRANCH} {REPO_URL}
        %cd {REPO_DIR}
    else:
        %cd {REPO_DIR}
        !git pull
else:
    !git pull

!pip install -q -r requirements.txt
!python -m src.data.download --output-dir data

# Tự động lấy WANDB_API_KEY từ Colab Secrets / Kaggle Secrets; nếu không có thì chạy offline
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("WANDB_API_KEY")
        if key:
            os.environ["WANDB_API_KEY"] = key
    except Exception:
        pass

if os.environ.get("WANDB_API_KEY"):
    os.environ.pop("WANDB_MODE", None)
    print("WANDB_API_KEY loaded -> logging online to Weights & Biases")
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_API_KEY not set -> running in WANDB_MODE=offline")

Cloning into 'NER-PhoNER_COVID19-UET'...
remote: Enumerating objects: 357, done.
remote: Counting objects: 100% (357/357), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 357 (delta 147), reused 319 (delta 120), pack-reused 0 (from 0)
Receiving objects: 100% (357/357), 1.15 MiB | 34.65 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/NER-PhoNER_COVID19-UET
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
[download] Đang tải: https://github.com/VinAIResearch/PhoNER_COVID19/archive/refs/heads/main.zip
[download] Đã tải 3.60 MB, đang giải nén thư mục data/ ...
[download] Hoàn tất: /content/NER-PhoNER_COVID19-UET/data
[download] Kiểm tra dữ liệu:
  word     train:  5027 câu | 132511 token
  word     dev  :  2000 câu |  56283 token
  word     test :  3000 câu |  85678 token
  syllable train:  5027 câu | 167541 token
  syllable d

In [2]:
# 1. Mô hình đầy đủ: BiLSTM + CharCNN + CRF
!python scripts/train.py --config configs/bilstm_crf.yaml --override data.level=syllable experiment_name=bilstm_crf_syllable training.checkpoint_dir=checkpoints/bilstm_crf_syllable

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttins (SuperMario-UET) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run cfvc15pc (0.3s)
wandb: ⣯ setting up run cfvc15pc (0.3s)
wandb: ⣟ setting up run cfvc15pc (0.3s)
wandb: ⡿ setting up run cfvc15pc (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/NER-PhoNER_COVID19-UET/wandb/run-20261121_095019-cfvc15pc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm_crf_syllable
wandb: ⭐️ View project at https://wandb.ai/NER-UET/ner-phoner-covid19-uet
wandb: 🚀 View run at https://wandb.ai/NER-UET/ner-phoner-covid19-uet/runs/cfvc15pc
[step 1] epoch=1.0000, train_loss=22.0103, dev_micro_f1=0.7402, dev_macro_f1=0.5299, dev

In [3]:
# 2. Ablation 1: Tắt tầng CRF (BiLSTM + CharCNN + Softmax)
!python scripts/train.py --config configs/bilstm_crf.yaml --override data.level=syllable experiment_name=bilstm_softmax_syllable model.use_crf=false training.checkpoint_dir=checkpoints/bilstm_softmax_syllable

# 3. Ablation 2: Tắt tầng Character Embedding (BiLSTM + CRF)
!python scripts/train.py --config configs/bilstm_crf.yaml --override data.level=syllable experiment_name=bilstm_crf_nochar_syllable model.use_char=false training.checkpoint_dir=checkpoints/bilstm_crf_nochar_syllable

# 4. Ablation 3: Thay CharCNN bằng CharBiLSTM
!python scripts/train.py --config configs/bilstm_crf.yaml --override data.level=syllable experiment_name=bilstm_charlstm_crf_syllable model.char_encoder_type=bilstm training.checkpoint_dir=checkpoints/bilstm_charlstm_crf_syllable

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttins (SuperMario-UET) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run mj6gpmwd (0.3s)
wandb: ⣯ setting up run mj6gpmwd (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/NER-PhoNER_COVID19-UET/wandb/run-20261121_095752-mj6gpmwd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm_softmax_syllable
wandb: ⭐️ View project at https://wandb.ai/NER-UET/ner-phoner-covid19-uet
wandb: 🚀 View run at https://wandb.ai/NER-UET/ner-phoner-covid19-uet/runs/mj6gpmwd
[step 1] epoch=1.0000, train_loss=0.7513, dev_micro_f1=0.6898, dev_macro_f1=0.4874, dev_bio_violations=1236.0000
[step 2] epoch=2.0000, train_loss=0.2962, dev_micro

In [4]:
import json

for path in sorted(Path("outputs/metrics").glob("bilstm*syllable*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    t = data["test"]
    print(
        f"{data['experiment_name']:28s} | P={t['micro_precision']:.4f} | "
        f"R={t['micro_recall']:.4f} | micro-F1={t['micro_f1']:.4f} | "
        f"macro-F1={t['macro_f1']:.4f} | BIO_viol={t['bio_violations']}"
    )

bilstm_charlstm_crf_syllable | P=0.9328 | R=0.9251 | micro-F1=0.9289 | macro-F1=0.8944 | BIO_viol=32
bilstm_crf_nochar_syllable   | P=0.9207 | R=0.9066 | micro-F1=0.9136 | macro-F1=0.8633 | BIO_viol=39
bilstm_crf_syllable          | P=0.9283 | R=0.9174 | micro-F1=0.9229 | macro-F1=0.8870 | BIO_viol=37
bilstm_softmax_syllable      | P=0.9194 | R=0.9179 | micro-F1=0.9186 | macro-F1=0.8802 | BIO_viol=462


In [5]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/NER-UET-outputs/bilstm_syllable
!cp outputs/metrics/bilstm*syllable*.json /content/drive/MyDrive/NER-UET-outputs/bilstm_syllable/
!cp -r checkpoints /content/drive/MyDrive/NER-UET-outputs/bilstm_syllable/
!ls -lh /content/drive/MyDrive/NER-UET-outputs/bilstm_syllable

Mounted at /content/drive
total 44K
-rw------- 1 root root 9.7K Nov 21 10:24 bilstm_charlstm_crf_syllable.json
-rw------- 1 root root  11K Nov 21 10:24 bilstm_crf_nochar_syllable.json
-rw------- 1 root root 8.1K Nov 21 10:24 bilstm_crf_syllable.json
-rw------- 1 root root  11K Nov 21 10:24 bilstm_softmax_syllable.json
drwx------ 6 root root 4.0K Nov 21 10:24 checkpoints
